# Retrieval-Augmented Generation (RAG)

### Goal

Build a simple RAG pipeline from scratch: embed documents, index them with FAISS, retrieve relevant passages for a query, and generate an answer using a local language model.

### What is RAG?

**Retrieval-Augmented Generation** combines two steps:
1. **Retrieve** relevant documents from a knowledge base
2. **Generate** an answer using an LLM, grounded in the retrieved context

### Why RAG?

LLMs have a fixed knowledge cutoff, tend to hallucinate facts, and cannot access private or domain-specific data. RAG addresses all three issues by providing the model with up-to-date, relevant context at inference time.

### Overview of this notebook

1. Load a Wikipedia passage corpus and QA pairs
2. Embed passages with a sentence-transformer model
3. Index embeddings with FAISS for fast similarity search
4. Build a TF-IDF baseline for comparison
5. Combine retrieval with a local LLM (SmolLM2) to answer questions
6. Evaluate retrieval and generation quality

### Sources

- [HuggingFace LLM Course -- Semantic Search with FAISS](https://huggingface.co/learn/llm-course/en/chapter5/6)
- [Sentence-Transformers documentation](https://www.sbert.net)

In [ ]:
# Install required packages (if not already installed)
%pip install -q datasets sentence-transformers faiss-cpu transformers torch scikit-learn matplotlib

## 1. Loading the Dataset

We use the `rag-datasets/rag-mini-wikipedia` dataset from HuggingFace, which contains:
- A **text corpus** of ~3,200 Wikipedia passages
- A **question-answer** set of 918 QA pairs with ground-truth answers

This is a convenient dataset for experimenting with RAG because the passages are short and the questions have known answers.

In [ ]:
from datasets import load_dataset

# Load the text corpus (Wikipedia passages)
corpus = load_dataset(
    "rag-datasets/rag-mini-wikipedia", "text-corpus"
)["passages"]

# Load the question-answer pairs
qa = load_dataset(
    "rag-datasets/rag-mini-wikipedia", "question-answer"
)["test"]

print(f"Corpus: {len(corpus)} passages")
print(f"QA set: {len(qa)} question-answer pairs")

In [ ]:
# Explore the corpus: look at the first few passages
for i in range(3):
    passage = corpus[i]["passage"]
    print(f"--- Passage {i} ({len(passage)} chars) ---")
    print(passage[:200], "...")
    print()

In [ ]:
# Compute some statistics about the corpus
import numpy as np

lengths = [len(p["passage"]) for p in corpus]
print(f"Number of passages: {len(lengths)}")
print(f"Average passage length: {np.mean(lengths):.0f} characters")
print(f"Min: {np.min(lengths)}, Max: {np.max(lengths)}")
print(f"Median: {np.median(lengths):.0f}")

In [ ]:
# Explore the QA pairs
for i in range(5):
    print(f"Q: {qa[i]['question']}")
    print(f"A: {qa[i]['answer']}")
    print()

## 2. Text Embeddings with Sentence-Transformers

Text embeddings are **dense vector representations** that capture the semantic meaning of text. Similar texts will have similar vectors (high cosine similarity), even if they use different words.

We use `sentence-transformers/all-MiniLM-L6-v2`, a lightweight model (22M parameters) that produces 384-dimensional vectors. It is fast enough to embed thousands of passages on a CPU in under a minute.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
# Embed a few example sentences
example_sentences = [
    "The capital of France is Paris.",
    "Paris is the largest city in France.",
    "Machine learning is a branch of artificial intelligence.",
    "The weather is sunny today.",
]

example_embeddings = embed_model.encode(example_sentences)
print(f"Embeddings shape: {example_embeddings.shape}")
print(f"First embedding (first 10 dims): {example_embeddings[0][:10]}")

In [ ]:
# Compute cosine similarity between all pairs
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(example_embeddings)

print("Cosine similarity matrix:")
for i, s1 in enumerate(example_sentences):
    for j, s2 in enumerate(example_sentences):
        if j > i:
            print(f"  {sim_matrix[i, j]:.3f}  '{s1[:40]}' <-> '{s2[:40]}'")

Notice that semantically related sentences (both about Paris/France) have higher similarity than unrelated ones (Paris vs weather).

In [ ]:
# Visualize pairwise similarity of 20 corpus passages as a heatmap
import matplotlib.pyplot as plt

n_samples = 20
sample_passages = [corpus[i]["passage"] for i in range(n_samples)]
sample_embeddings = embed_model.encode(sample_passages)
sim = cosine_similarity(sample_embeddings)

plt.figure(figsize=(8, 6))
plt.imshow(sim, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Cosine similarity")
plt.title(f"Pairwise similarity of {n_samples} corpus passages")
plt.xlabel("Passage index")
plt.ylabel("Passage index")
plt.tight_layout()
plt.show()

## 3. Indexing with FAISS

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search over large collections of vectors. Instead of computing cosine similarity against every passage (O(n) per query), FAISS can use approximate nearest neighbor algorithms to search much faster.

For our small corpus (~3.2k passages), we use `IndexFlatIP` (exact inner product search). For larger collections, FAISS supports approximate methods like IVF and HNSW.

Since we normalize our embeddings, inner product equals cosine similarity.

In [ ]:
import faiss

# Quick demo: build a small index and search it
demo_dim = sample_embeddings.shape[1]
demo_index = faiss.IndexFlatIP(demo_dim)

# Normalize embeddings for cosine similarity via inner product
faiss.normalize_L2(sample_embeddings)
demo_index.add(sample_embeddings)
print(f"Demo index contains {demo_index.ntotal} vectors of dimension {demo_dim}")

In [ ]:
# Search the demo index with a query
query = "Who was the first president of the United States?"
query_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")

scores, indices = demo_index.search(query_emb, k=3)

print(f"Query: {query}\n")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
    print(f"  {rank+1}. (score={score:.3f}) {sample_passages[idx][:120]}...")

## 4. Exercise: Build the Embedding Index

Now it is your turn. Complete the following tasks:

1. Embed **all** corpus passages using `embed_model.encode()` with `normalize_embeddings=True` and `batch_size=64`
2. Build a FAISS `IndexFlatIP` index and add all embeddings
3. Write a `search(query, k=5)` function that embeds a query, searches the index, and returns the top-k passages with their scores
4. Test it on 3 sample questions

In [ ]:
# TODO: Embed all corpus passages and build the FAISS index.
# Write a search(query, k=5) function that returns top-k passages
# with scores. Test it on 3 sample questions.

passages = corpus["passage"]
print(f"Number of passages to embed: {len(passages)}")

In [ ]:
# %load solutions/embed_and_index.py

## 5. Baseline: TF-IDF Retrieval

Before celebrating the power of semantic search, let us build a simple **TF-IDF** baseline using scikit-learn. TF-IDF is a classic keyword-based retrieval method: it represents documents as sparse vectors of term frequencies, weighted by how rare each term is across the corpus.

This lets us compare: does semantic (dense) retrieval actually outperform keyword (sparse) retrieval?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# Build TF-IDF index over all passages
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000, stop_words="english"
)
tfidf_matrix = tfidf_vectorizer.fit_transform(passages)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


def tfidf_search(query, k=5):
    """Search using TF-IDF cosine similarity."""
    query_vec = tfidf_vectorizer.transform([query])
    similarities = sklearn_cosine(query_vec, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    results = []
    for idx in top_indices:
        results.append({
            "passage": passages[idx],
            "score": float(similarities[idx]),
            "index": int(idx),
        })
    return results

In [ ]:
# Compare semantic search vs TF-IDF on the same queries
comparison_queries = [
    "What is the capital of France?",
    "Who invented the telephone?",
    "How do earthquakes happen?",  # paraphrase: "What causes earthquakes?"
]

for query in comparison_queries:
    print(f"\nQuery: {query}")
    print("--- Dense (FAISS) ---")
    for r in search(query, k=2):
        print(f"  (score={r['score']:.3f}) {r['passage'][:100]}...")
    print("--- TF-IDF ---")
    for r in tfidf_search(query, k=2):
        print(f"  (score={r['score']:.3f}) {r['passage'][:100]}...")

Semantic search tends to work better for **paraphrases** and **synonym-rich** queries, because it matches meaning rather than exact keywords. TF-IDF can be better when the query contains rare, distinctive terms that appear verbatim in the documents.

## 6. The RAG Pipeline

Now we combine retrieval with generation. The RAG pipeline works as follows:

1. **Embed** the user question
2. **Retrieve** the top-k most relevant passages from FAISS
3. **Build a prompt** containing the retrieved context and the user question
4. **Generate** an answer using a local language model

We use **SmolLM2-360M-Instruct** from HuggingFace as our generation model. It is small enough to run on a CPU without any API key. Because it is a very small model, the generated answers will often be imperfect -- this is expected and illustrates why production RAG systems use larger models or API-based LLMs.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

gen_model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name, dtype=torch.float16
).to("cpu")
gen_model.eval()
print(f"Generation model loaded: {gen_model_name}")

In [ ]:
# Quick sanity check: generate a response without RAG context
test_messages = [
    {"role": "user", "content": "What is the capital of France?"}
]
inputs = tokenizer.apply_chat_template(
    test_messages, return_tensors="pt", return_dict=True
)
input_ids = inputs["input_ids"]
with torch.no_grad():
    output = gen_model.generate(**inputs, max_new_tokens=50, do_sample=False)
response = tokenizer.decode(
    output[0][input_ids.shape[1]:], skip_special_tokens=True
)
print(f"Model response (no context): {response}")

In [ ]:
# Full RAG example: retrieve context then generate
question = "Who invented the telephone?"

# Step 1: Retrieve top-3 passages
results = search(question, k=3)
print("Retrieved passages:")
for i, r in enumerate(results):
    print(f"  [{i+1}] (score={r['score']:.3f}) {r['passage'][:100]}...")

# Step 2: Build context string
context = "\n\n".join(
    f"[Passage {i+1}]: {r['passage']}" for i, r in enumerate(results)
)

# Step 3: Build prompt with system instruction + context + question
messages = [
    {"role": "system", "content": (
        "Answer the question based on the provided context. "
        "Be concise and specific. If the context doesn't contain "
        "the answer, say so."
    )},
    {"role": "user", "content": (
        f"Context:\n{context}\n\nQuestion: {question}"
    )},
]

# Step 4: Generate
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", return_dict=True
)
input_ids = inputs["input_ids"]
with torch.no_grad():
    output = gen_model.generate(**inputs, max_new_tokens=100, do_sample=False)
answer = tokenizer.decode(
    output[0][input_ids.shape[1]:], skip_special_tokens=True
)

print(f"\nQuestion: {question}")
print(f"RAG Answer: {answer}")

### What happens when retrieval fails?

If the retrieved passages are not relevant to the question, the model has no useful context and will either hallucinate or produce a generic response. The quality of retrieval is therefore critical to the overall RAG pipeline.

**Note on model size:** SmolLM2-360M is a very small model. It will sometimes produce incorrect, incomplete, or oddly formatted answers even when the retrieved context is correct. In production RAG systems, one would typically use a much larger model (7B+ parameters) or an API-based model (GPT-4, Claude, etc.).

## 7. Exercise: Complete RAG Query Function

Implement a `rag_answer(question, k=3)` function that performs the full retrieve-then-generate pipeline:

1. Retrieve top-k passages using the `search()` function
2. Build a context string from the retrieved passages
3. Construct a chat prompt with a system message and the context + question
4. Generate an answer with `gen_model`
5. Return a dictionary with the answer and the retrieved passages

Test your function on 5 questions from the QA set.

In [ ]:
# TODO: Implement rag_answer(question, k=3) and test on 5 QA pairs


In [ ]:
# %load solutions/rag_query.py

## 8. Evaluation

We have ground-truth answers for 918 questions. Let us use them to evaluate both the **retrieval** and **generation** stages of our pipeline.

- **Retrieval evaluation**: For each question, check if the key terms from the ground-truth answer appear in the top-k retrieved passages. This is a proxy for Recall@k.
- **Generation evaluation**: Check if the key terms from the ground-truth answer appear in the generated answer. This is a simple heuristic -- production systems use more sophisticated metrics like ROUGE, BERTScore, or human evaluation.

We evaluate on a subset of 50 questions for speed.

In [ ]:
# Quick look at the evaluation we'll do
example_q = qa[0]["question"]
example_a = qa[0]["answer"]
print(f"Question: {example_q}")
print(f"Ground truth answer: {example_a}")
print()

# Show what key terms we extract from the answer
stop_words = {"the", "a", "an", "is", "was", "of", "in", "to", "and", "for", "on", "it"}
gt_terms = set(example_a.lower().split()) - stop_words
print(f"Key terms from ground truth: {gt_terms}")

## 9. Exercise: Evaluate the RAG Pipeline

Compute the following metrics over 50 QA pairs:

1. **Dense retrieval Recall@5**: fraction of questions where key answer terms appear in the top-5 FAISS-retrieved passages
2. **TF-IDF retrieval Recall@5**: same metric using TF-IDF retrieval
3. **Answer key-term match**: fraction of questions where key answer terms appear in the RAG-generated answer

Compare dense vs TF-IDF retrieval performance.

In [ ]:
# TODO: Compute Recall@5 for dense and TF-IDF retrieval,
# and answer accuracy over 50 QA pairs.


In [ ]:
# %load solutions/evaluate_rag.py

## 10. Improving RAG

Our simple pipeline works, but there are many ways to improve it in practice:

### Chunking strategies

What if documents are too long to embed as a single vector? Split them into **chunks** with overlap:

```python
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks
```

Trade-off: smaller chunks are more precise but lose context; larger chunks preserve context but may dilute the relevant signal.

### Hybrid search

Combine dense (FAISS) and sparse (BM25/TF-IDF) retrieval. Dense search handles synonyms and paraphrases; sparse search handles rare keywords and exact matches. Many production systems use both and merge the results (Reciprocal Rank Fusion).

### Reranking

After retrieving top-k candidates with a fast bi-encoder (like our sentence-transformer), **rerank** them with a slower but more accurate **cross-encoder** model that sees the query and passage together:

```python
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict([(query, passage) for passage in candidates])
```

### Production tools

- **ChromaDB**: embedded vector database, easy to use for prototyping
- **LangChain / LlamaIndex**: frameworks that orchestrate RAG pipelines with many retrieval and generation options
- **pgvector**: PostgreSQL extension for vector similarity search

## 11. Going Further

### Advanced RAG techniques
- [NirDiamant/RAG_Techniques](https://github.com/NirDiamant/RAG_Techniques): comprehensive collection of advanced RAG methods
- [LangChain RAG from Scratch](https://github.com/langchain-ai/rag-from-scratch): step-by-step RAG implementation

### Evaluation frameworks
- [RAGAS](https://github.com/explodinggradients/ragas): automated RAG evaluation (faithfulness, answer relevancy, context precision)
- [DeepEval](https://github.com/confident-ai/deepeval): LLM evaluation framework with RAG-specific metrics

### Production considerations
- **Chunking**: experiment with different chunk sizes and overlap values for your data
- **Caching**: cache embeddings and frequent queries to reduce latency
- **Hybrid search**: combine dense and sparse retrieval for robustness
- **Larger models**: use 7B+ parameter models or API-based LLMs for better generation quality